# Transformers

In [ ]:
from seismi.xls import read_seismic_events, read_interpolated

seismic_events = read_seismic_events('SL jevy pri dobyvaní R 140 704.xlsx')
depths_interpolated = read_interpolated('sloj 40 hloubka grid.xlsx')

In [ ]:
from seismi.dxf import load_dxf

mines_map = load_dxf('mines_map.dxf')

In [ ]:
from seismi.transformers import (
    SeismicTransformer, SeismicEventDataset, prepare_time_features,
    train_transformer_model, predict_future_events, evaluate_predictions,
    plot_predictions
)

seismic_events_time = prepare_time_features(seismic_events)

In [ ]:
from scipy.interpolate import griddata
import numpy as np

# Get coordinates and depth from depths_interpolated
depth_points = np.array(list(zip(depths_interpolated.geometry.x, depths_interpolated.geometry.y)))
depth_values = depths_interpolated['depth'].values

# Get seismic event coordinates
event_points = np.array(list(zip(seismic_events_time.geometry.x, seismic_events_time.geometry.y)))

# Interpolate depth at each seismic event location
event_depths = griddata(depth_points, depth_values, event_points, method='linear')

# Add depth column to seismic_events_time
seismic_events_time['depth'] = event_depths

# Check for and handle any NaN values in the interpolated depths
if np.isnan(event_depths).any():
    print(f"Warning: {np.isnan(event_depths).sum()} events have NaN depths. Filling with mean depth.")
    mean_depth = np.nanmean(event_depths)
    seismic_events_time['depth'] = seismic_events_time['depth'].fillna(mean_depth)

print(f"Depths added to seismic events data: Min={seismic_events_time['depth'].min():.2f}, Max={seismic_events_time['depth'].max():.2f}")

In [ ]:
import torch

# Parameters
seq_length = 10  # Use 10 consecutive events as input
prediction_horizon = 1  # Predict the next event
batch_size = 32

# Create dataset
event_dataset = SeismicEventDataset(
    seismic_events_time,
    seq_length=seq_length,
    prediction_horizon=prediction_horizon
)

# Split into train and validation sets (80/20)
train_size = int(0.8 * len(event_dataset))
val_size = len(event_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(event_dataset, [train_size, val_size])

# Create data loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size
)

print(f"Training sequences: {len(train_dataset)}")
print(f"Validation sequences: {len(val_dataset)}")

**Note:** The energy values are log-transformed before scaling and training. This ensures the model predicts positive energies and matches the distribution of the training data. After prediction, the inverse transform (`np.expm1`) is applied to recover the original energy scale.

In [ ]:
# Create model
model = SeismicTransformer(
    input_dim=5,  # X, Y, depth, time, energy
    embedding_dim=64,
    num_heads=4,
    num_encoder_layers=2,
    output_dim=5  # X, Y, depth, time, energy
)

# Train the model
history = train_transformer_model(
    model,
    train_loader,
    val_loader,
    epochs=5,
    lr=0.001
)

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Get a validation sequence for prediction
val_sequence, val_target = val_dataset[0]
val_sequence = val_sequence.unsqueeze(0)  # Add batch dimension

# Predict future events
n_future = 10
future_events = predict_future_events(
    model,
    val_sequence,
    n_future=n_future,
    scaler=event_dataset.scaler
)

# Get actual future events for comparison
# Note: This is just for demonstration, in a real scenario you'd use actual future data
actual_events = event_dataset.data[len(train_dataset):len(train_dataset)+n_future, :5]

# Evaluate predictions
metrics = evaluate_predictions(actual_events, future_events)
print("Prediction Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

# Plot predictions
fig = plot_predictions(actual_events, future_events, base_map=mines_map)
plt.show()

In [ ]:
from ipyleaflet import Map, CircleMarker, Popup
from ipywidgets import HTML
import numpy as np
from branca.colormap import linear
from datetime import timedelta

# Create a colormap for energy values
energy_min = min(actual_events[:, 4].min(), future_events[:, 4].min())
energy_max = max(actual_events[:, 4].max(), future_events[:, 4].max())
colormap = linear.viridis.scale(energy_min, energy_max)

# Calculate the center for the map
center_lat = np.mean(np.concatenate([actual_events[:, 1], future_events[:, 1]]))
center_lon = np.mean(np.concatenate([actual_events[:, 0], future_events[:, 0]]))

# Convert days_elapsed to datetime
origin_date = seismic_events['Datetime'].min()
def days_to_dt(days):
    return (origin_date + timedelta(days=float(days))).strftime('%Y-%m-%d %H:%M')

def add_event_markers(events, color, label):
    for i, row in enumerate(events):
        x, y, depth, days_elapsed, energy = row
        dt = days_to_dt(days_elapsed)
        popup = Popup(
            location=(y, x),
            child=HTML(f"{label} Event {i+1}<br>Energy: {energy:.2f}<br>Depth: {depth:.2f}<br>Date: {dt}"),
            close_button=False
        )
        marker = CircleMarker(
            location=(y, x),
            radius=8,
            color=color,
            fill_color=colormap(energy),
            fill_opacity=0.7,
            weight=2,
            popup=popup
        )
        m.add_layer(marker)

# Create map
m = Map(center=(center_lat, center_lon), zoom=14)

# Add markers for actual and predicted events
add_event_markers(actual_events, 'blue', 'True')
add_event_markers(future_events, 'red', 'Predicted')

# Sanity check: print min/max of predicted energies
print(f"Predicted energies: min={future_events[:,4].min():.2f}, max={future_events[:,4].max():.2f}")

m

In [ ]:
from ipyleaflet import Map, CircleMarker, Popup, LegendControl
from ipywidgets import HTML
from branca.colormap import linear
import numpy as np
from datetime import timedelta

# Sample a subset for clarity
n_train = min(100, len(train_dataset))
n_test = min(100, len(val_dataset))
n_pred = min(10, len(future_events))

# Get indices
train_indices = np.random.choice(len(train_dataset), n_train, replace=False)
test_indices = np.arange(n_test)

# Unscale all features for train/test
train_unscaled = event_dataset.scaler.inverse_transform(event_dataset.data_scaled[train_indices])
test_unscaled = event_dataset.scaler.inverse_transform(event_dataset.data_scaled[len(train_dataset):len(train_dataset)+n_test])
pred_unscaled = future_events[:n_pred]

# All arrays: columns are [X, Y, depth, days_elapsed, energy]
groups = [
    (train_unscaled, 'navy', 'Train'),
    (test_unscaled, 'darkgreen', 'Test'),
    (pred_unscaled, 'darkred', 'Prediction'),
]

# Colormap and size scaling
all_energies = np.concatenate([g[0][:,4] for g in groups])
colormap = linear.YlOrRd_09.scale(all_energies.min(), all_energies.max())
size_scale = lambda e: 6 + 14 * (e - all_energies.min()) / (all_energies.max() - all_energies.min() + 1e-6)

# Map center
all_x = np.concatenate([g[0][:,0] for g in groups])
all_y = np.concatenate([g[0][:,1] for g in groups])
center_lat = np.mean(all_y)
center_lon = np.mean(all_x)
m = Map(center=(center_lat, center_lon), zoom=14)

# Convert days_elapsed to datetime
origin_date = seismic_events_time['Datetime'].min()
def days_to_dt(days):
    return (origin_date + timedelta(days=float(days))).strftime('%Y-%m-%d %H:%M')

# Helper to add markers for a group
for arr, color, label in groups:
    for row in arr:
        x, y, depth, days_elapsed, energy = row
        dt = days_to_dt(days_elapsed)
        popup = Popup(
            location=(y, x),
            child=HTML(
                f"<b>{label}</b><br>"
                f"Energy: {energy:.2f}<br>"
                f"Depth: {depth:.2f}<br>"
                f"Date: {dt}"
            ),
        )
        marker = CircleMarker(
            location=(y, x),
            radius=int(size_scale(energy)),
            color=color,
            fill_color=colormap(energy),
            fill_opacity=0.8 if label == "Prediction" else 0.7,
            weight=2,
            popup=popup
        )
        m.add_layer(marker)

# Add a legend for color scale
colormap.caption = 'Seismic Energy'
m.add_control(LegendControl({
    'Training': 'navy',
    'Test': 'darkgreen',
    'Prediction': 'darkred',
}, name='Legend', position='topright'))
m.add_control(colormap)

m